# Incremental Data Pipelines (Avoid Reprocessing Data)


## Why Incremental Pipelines Matter

**Bad pipeline:**
- Reprocesses all data every run
- Slow
- Expensive

**Good pipeline:**
- Processes only new/updated data
- Fast
- Scalable

👉 How production systems stay efficient


## Incremental strategy

Track **high-water marks** — last processed **ID** or **timestamp** — then:

👉 Fetch **only** rows after that mark.


## Prerequisites

- PostgreSQL running ([`../02_sql/README.md`](../02_sql/README.md)).
- This notebook is **self-contained**: creates `api_users` and `pipeline_state` (same `api_users` shape as notebook `03_building_etl_pipeline.ipynb`).
- Network for JSONPlaceholder.


## Connect + tables + initial state


In [ ]:
import psycopg2
import requests

conn = psycopg2.connect(
    host="localhost",
    database="de_course",
    user="admin",
    password="admin",
)

cur = conn.cursor()

cur.execute(
    """
CREATE TABLE IF NOT EXISTS api_users (
    id INT PRIMARY KEY,
    name TEXT,
    email TEXT,
    city TEXT
)
"""
)

cur.execute(
    """
CREATE TABLE IF NOT EXISTS pipeline_state (
    id SERIAL PRIMARY KEY,
    last_processed_id INT
)
"""
)

conn.commit()

cur.execute("SELECT * FROM pipeline_state")
if not cur.fetchone():
    cur.execute("INSERT INTO pipeline_state (last_processed_id) VALUES (0)")
    conn.commit()
    print("Initialized pipeline_state with last_processed_id = 0")
else:
    print("pipeline_state already has rows — skipped init insert.")


## Get last processed ID


In [ ]:
def get_last_processed_id():
    cur.execute("SELECT last_processed_id FROM pipeline_state LIMIT 1")
    return cur.fetchone()[0]


## Extract only new data


In [ ]:
def extract_incremental(last_id):
    url = "https://jsonplaceholder.typicode.com/users"
    data = requests.get(url).json()

    return [d for d in data if d["id"] > last_id]


## Transform + load (same pattern as notebook 03)


In [ ]:
def transform(data):
    cleaned = []

    for d in data:
        try:
            cleaned.append(
                {
                    "id": d["id"],
                    "name": d["name"].strip().title(),
                    "email": d["email"].lower(),
                    "city": d["address"]["city"],
                }
            )
        except Exception:
            continue

    return cleaned


def load(data):
    for d in data:
        try:
            cur.execute(
                """
            INSERT INTO api_users (id, name, email, city)
            VALUES (%s, %s, %s, %s)
            ON CONFLICT (id) DO NOTHING
            """,
                (d["id"], d["name"], d["email"], d["city"]),
            )
        except Exception as e:
            print("Insert failed:", e)

    conn.commit()


## Full incremental pipeline


In [ ]:
def run_incremental_pipeline():
    last_id = get_last_processed_id()

    new_data = extract_incremental(last_id)

    if not new_data:
        print("No new data")
        return

    cleaned = transform(new_data)
    load(cleaned)

    max_id = max(d["id"] for d in new_data)
    update_state(max_id)

    print(f"Processed {len(new_data)} new records")


def update_state(new_last_id):
    cur.execute(
        """
    UPDATE pipeline_state
    SET last_processed_id = %s
    """,
        (new_last_id,),
    )
    conn.commit()


run_incremental_pipeline()


## Validate


In [ ]:
cur.execute("SELECT * FROM pipeline_state")
print("pipeline_state:", cur.fetchall())

cur.execute("SELECT COUNT(*) FROM api_users")
print("api_users count:", cur.fetchone()[0])


## Practice

1. Run **`run_incremental_pipeline()`** twice.
2. Confirm the **second** run prints **`No new data`** (watermark past max API id).


## Assignment

Upgrade the incremental system:

- Track **timestamp** (`updated_at`) instead of (or in addition to) numeric id
- Handle **updates** to existing rows (not only inserts)
- **Log** processed record counts and watermark

**Bonus:** batch with `LIMIT` / paging on the API side.


## Interview Questions

1. What is incremental processing?
2. Why is it important?
3. How do you track processed data?
4. How do you handle updates vs inserts?


## What you just built

- **Stateful** high-water-mark pipeline
- **Efficient** re-runs (skip already-seen keys)

👉 Same idea as scheduled Airflow/Spark jobs — smaller demo.


---

**Next:** **ETL project** — combine extract/clean/load/incremental into a portfolio piece.


## Optional: close connection


In [ ]:
cur.close()
conn.close()
print("Connection closed.")


## Your tasks

- [ ] Fresh kernel: run **Connect + tables** → **Full incremental pipeline** → **Validate** → run **`run_incremental_pipeline()`** again.
- [ ] Confirm **second run** processes **0** new users (watermark ≥ 10 for JSONPlaceholder).
- [ ] **Assignment:** timestamp tracking; upserts for updates; logging; **bonus:** batch size.
- [ ] Answer interview questions; compare **id watermark** vs **CDC / binlog** for updates.
